# GTZAN Initial EDA

This notebook visualizes the local GTZAN dataset that lives at `data/genres_original`.

It covers:
- genre class distribution
- basic audio metadata checks
- one MFCC example per genre

## Environment Setup

Run this first. It finds the project root, points Python at the `code/` folder, and installs missing requirements into the active notebook kernel if needed.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import importlib.util
import os
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    """Find the audio-genre-classifier repo from common notebook cwd values."""
    start = start.resolve()
    candidates = []
    for path in (start, *start.parents):
        candidates.append(path)
        candidates.append(path / "audio-genre-classifier")

    for candidate in candidates:
        if (candidate / "code" / "genre_distribution_eda.py").exists():
            return candidate

    raise FileNotFoundError("Could not find the audio-genre-classifier repo root.")


repo_root = find_repo_root(Path.cwd())
os.chdir(repo_root)

code_dir = repo_root / "code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".matplotlib-cache"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(repo_root / ".numba-cache"))

required_modules = ["matplotlib", "pandas", "librosa"]
missing_modules = [
    module for module in required_modules if importlib.util.find_spec(module) is None
]

if missing_modules:
    print(f"Installing missing modules into this kernel: {missing_modules}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(repo_root / "requirements.txt")]
    )
else:
    print("Notebook kernel already has the required modules.")

dataset_path = repo_root / "data" / "genres_original"
report_dir = repo_root / "reports"
report_dir.mkdir(exist_ok=True)
print(f"Project root: {repo_root}")
print(f"Dataset path: {dataset_path}")

In [ ]:
from genre_distribution_eda import (
    build_audio_metadata,
    build_genre_counts,
    plot_genre_distribution,
    validate_gtzan_layout,
)
from mfcc_examples_eda import plot_mfcc_examples

## Genre Class Distribution

GTZAN should have 10 genre folders with 100 `.wav` files per genre.

In [ ]:
genre_counts = build_genre_counts(dataset_path)
display(genre_counts)

In [ ]:
layout_validation = validate_gtzan_layout(genre_counts)
display(layout_validation)

In [ ]:
genre_chart_path = report_dir / "notebook_genre_distribution.png"
plot_genre_distribution(genre_counts, output_path=genre_chart_path)
import matplotlib.pyplot as plt
plt.close("all")
display(Image(filename=str(genre_chart_path)))

## Audio Metadata Smoke Check

This loads a small sample with `librosa` so the notebook stays quick while still confirming audio files can be read.

In [ ]:
metadata_sample = build_audio_metadata(dataset_path, limit=20)
display(metadata_sample.head())

In [ ]:
metadata_summary = (
    metadata_sample.groupby("genre")
    .agg(
        files=("file_name", "count"),
        avg_duration_seconds=("duration_seconds", "mean"),
        min_sample_rate=("sample_rate", "min"),
        max_sample_rate=("sample_rate", "max"),
    )
    .round(2)
)
display(metadata_summary)

## MFCC Examples By Genre

This renders one deterministic example file per genre using the first `.wav` file in each folder.

In [ ]:
mfcc_chart_path = report_dir / "notebook_mfcc_examples_by_genre.png"
plot_mfcc_examples(dataset_path=dataset_path, output_path=mfcc_chart_path)
display(Image(filename=str(mfcc_chart_path)))